# 10.2 REINFORCE와 실습 — 노트북

[![Open In Colab: REINFORCE](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter10_2_reinforce.ipynb)

책 본문: [10.2 REINFORCE와 실습](https://smhanlab.com/book-ml/kor/ml2/chapter10.html)

이 노트북은 책 10.2절의 REINFORCE 코드를 그대로 실행합니다:
(1) 2-arm 밴딧에서 "좋은 행동을 더 선호한다"는 것을 눈으로 확인하고,
(2) PyTorch + CartPole에서 REINFORCE를 학습시켜 리턴이 오르는 것을 봅니다.
(3) 리턴을 정규화(평균을 빼는 베이스라인)하지 않을 때와 비교합니다.


## 0. 설정: 한국어 폰트, 시드 고정, 이미지 저장 경로


In [1]:
import os, random, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한국어 라벨을 폰트 'Noto Sans CJK KR'으로 (없으면 기본 폰트로 넘어가도 출력은 됨)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print(f"그림 저장 위치: {IMG}")


그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 2-arm 밴딧: REINFORCE가 정말 "좋은 행동을 선호"하게 만드는가?

상태가 없는 가장 단순한 환경(10.2절 "밴딧 실험"):
행동 0 → 보상 `+1`, 행동 1 → 보상 `-1`(1스텝이므로 리턴 `G_t = r_t`).
책의 `softmax_policy`와 `reinforce_update`를 그대로 씁니다.
`theta=[0,0]`(두 행동 반반)에서 시작해 `alpha=0.05`로 300 에피소드 갱신하면,
**좋은 행동 0의 확률이 0.5 → 0.98로** 올라가는 것을 기대합니다.


In [2]:
def softmax_policy(theta, state_feature):
    logits = [theta[0]*state_feature, theta[1]*state_feature]
    m = max(logits)
    exps = [math.exp(l - m) for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

def reinforce_update(theta, episode, alpha, gamma):
    # episode: [(state_feature, action, reward), ...]
    T = len(episode)
    G = [0.0] * T
    running = 0.0
    for t in reversed(range(T)):
        running = episode[t][2] + gamma * running
        G[t] = running
    for t, (s, a, r) in enumerate(episode):
        probs = softmax_policy(theta, s)
        if a == 0:
            grad_log_pi = [(1 - probs[0]) * s, -probs[1] * s]
        else:
            grad_log_pi = [-probs[0] * s, (1 - probs[1]) * s]
        theta[0] += alpha * G[t] * grad_log_pi[0]
        theta[1] += alpha * G[t] * grad_log_pi[1]
    return theta


In [3]:
random.seed(0)
theta = [0.0, 0.0]
state_feature = 1.0
p0_hist = []
for ep in range(300):
    probs = softmax_policy(theta, state_feature)
    p0_hist.append(probs[0])
    a = 0 if random.random() < probs[0] else 1
    r = 1.0 if a == 0 else -1.0
    theta = reinforce_update(theta, [(state_feature, a, r)], alpha=0.05, gamma=0.99)

final = softmax_policy(theta, state_feature)
print("최종 theta =", [round(x, 4) for x in theta])
print("최종 p(행동 0) =", round(final[0], 4))
for k in [1, 50, 100, 200, 300]:
    print(f"  {k:3d}회 에피소드 후 p(행동 0) = {p0_hist[k-1]:.4f}")


최종 theta = [2.0352, -2.0352]
최종 p(행동 0) = 0.9832
    1회 에피소드 후 p(행동 0) = 0.5000
   50회 에피소드 후 p(행동 0) = 0.8719
  100회 에피소드 후 p(행동 0) = 0.9443
  200회 에피소드 후 p(행동 0) = 0.9727
  300회 에피소드 후 p(행동 0) = 0.9832


이 그래프가 책의 "smoke test"입니다: 확률이 0.5(무작위)에서 출발해
1에 점근하듯 0.98에 접근합니다. 대부분 상승(0.5→0.87)은 처음 50 에피소드에
집중됩니다(10.2절의 "느리게 수렴" 설명).


In [4]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 301), p0_hist, color="tab:blue", linewidth=1.2)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Episode"); ax.set_ylabel("p(action 0)")
ax.set_title("2-Arm Bandit: p(good action) under REINFORCE")
ax.set_ylim(0.4, 1.01); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch10_2_reinforce_bandit.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch10_2_reinforce_bandit.svg")


저장: /home/smhan/book-ml/kor/src/images/ch10_2_reinforce_bandit.svg


### 1.1 one-vs-rest 그래디언트를 수치적으로 검증

책의 "코드가 실제로 하는 일"에서 유도한 softmax 그래디언트 식

\[\frac{\partial \log \pi(0|s)}{\partial \theta} =
[(1-p_0)s,\; -p_1 s], \qquad
\frac{\partial \log \pi(1|s)}{\partial \theta} =
[-p_0 s,\; (1-p_1)s]\]

을, "확률함수를 `epsilon`만큼 직접 빼서" 계산한 **수치 미분**과 비교해
정확히 같은지 확인합니다. `epsilon`이 작을수록 두 값이 가까워야 합니다.


In [5]:
def numeric_grad(theta, s, action, eps=1e-6):
    def logp(th):
        p = softmax_policy(th, s)
        return math.log(p[action] + 1e-12)
    g = [0.0, 0.0]
    for i in range(2):
        thp = list(theta); thm = list(theta)
        thp[i] += eps; thm[i] -= eps
        g[i] = (logp(thp) - logp(thm)) / (2 * eps)
    return g

theta = [1.0, 0.0]; s = 1.0
probs = softmax_policy(theta, s)
for a in [0, 1]:
    if a == 0:
        analytic = [(1 - probs[0]) * s, -probs[1] * s]
    else:
        analytic = [-probs[0] * s, (1 - probs[1]) * s]
    num = numeric_grad(theta, s, a)
    ok = all(abs(an - nn) < 1e-5 for an, nn in zip(analytic, num))
    print(f"action {a}: analytic={['%.4f'%x for x in analytic]}  "
          f"numeric={['%.4f'%x for x in num]}  일치={ok}")


action 0: analytic=['0.2689', '-0.2689']  numeric=['0.2689', '-0.2689']  일치=True
action 1: analytic=['-0.7311', '0.7311']  numeric=['-0.7311', '0.7311']  일치=True


## 2. CartPole에서 REINFORCE (PyTorch 자동 미분)

책의 `PolicyNet` / `run_episode` / `reinforce_update`를 그대로 옮깁니다.
`torch.distributions.Categorical.log_prob()`가 10.1절의
`log pi_theta(a_t|s_t)`를, `.backward()`가 Policy Gradient Theorem의
그래디언트를 자동으로 계산합니다. **리턴 정규화(평균을 빼는 베이스라인)를
`normalize` 플래그로 켜고 끄며** 비교합니다.


In [6]:
import torch, torch.nn as nn
import gymnasium as gym

class PolicyNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, n_actions))
    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)

def run_episode(policy, env, seed):
    s, _ = env.reset(seed=seed)
    log_probs, rewards = [], []
    for _ in range(500):
        probs = policy(torch.tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))   # log pi_theta(a|s)
        s, r, term, trunc, _ = env.step(a.item())
        rewards.append(r)
        if term or trunc:
            break
    return log_probs, rewards

def reinforce_update(policy, opt, log_probs, rewards, gamma, normalize=True):
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    if normalize:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = -sum(lp * g for lp, g in zip(log_probs, returns))  # 부호 반전 = 경사 상승
    opt.zero_grad(); loss.backward(); opt.step()
    return loss

def train_reinforce(seed, n_episodes=300, gamma=0.99, normalize=True, lr=1e-2):
    torch.manual_seed(seed)
    env = gym.make("CartPole-v1")
    policy = PolicyNet(4, 2)          # 상태 4차원, 행동 2개
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    returns_all = []
    for ep in range(n_episodes):
        log_probs, rewards = run_episode(policy, env, seed + ep)
        returns_all.append(sum(rewards))
        reinforce_update(policy, opt, log_probs, rewards, gamma, normalize)
    env.close()
    return returns_all

def stats(rets):
    return sum(rets[:20]) / 20, sum(rets[-20:]) / 20, max(rets)


In [7]:
# 정규화(베이스라인) 있는 학습 — seed 0, 300 에피소드
returns_norm = train_reinforce(seed=0, normalize=True)
f, l, b = stats(returns_norm)
print(f"정규화 있음  (seed 0): 처음 20 에피소드 평균={f:.2f}  "
      f"마지막 20 에피소드 평균={l:.2f}  최고={b}")


정규화 있음  (seed 0): 처음 20 에피소드 평균=14.05  마지막 20 에피소드 평균=99.45  최고=500.0


### 2.1 학습 곡선

연회색이 에피소드별 리턴(노이즈가 큼), 파란 실선이 20 에피소드 이동
평균, 빨간 점선이 만점 500입니다. **이동 평균이 500에 포화되지 않고
100대 수준에서 멈추는 것**이 REINFORCE의 "느리고 불안정" 학습을 보여줍니다.


In [8]:
def moving_average(x, w):
    out = [sum(x[max(0, i-w+1):i+1]) / min(w, i+1) for i in range(len(x))]
    return out

ep = list(range(1, len(returns_norm)+1))
ma = moving_average(returns_norm, 20)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ep, returns_norm, color="lightgray", linewidth=0.8, label="per-episode return")
ax.plot(ep, ma, color="tab:blue", linewidth=1.6, label="20-episode moving average")
ax.axhline(500, color="tab:red", linestyle="--", linewidth=1, label="max score 500")
ax.set_xlabel("Episode"); ax.set_ylabel("Return")
ax.set_title("CartPole: REINFORCE training curve over 300 episodes")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch10_2_reinforce_cartpole_curve.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch10_2_reinforce_cartpole_curve.svg")


저장: /home/smhan/book-ml/kor/src/images/ch10_2_reinforce_cartpole_curve.svg


## 3. 자주 하는 실수: 리턴을 정규화(평균 빼기)하지 않을 때

책의 "자주 하는 실수"에서 짚은 `returns - returns.mean()`(= 에피소드당
평균 리턴을 빼는 가장 단순한 **베이스라인**)을 빼면, CartPole 리턴이
항상 양수라 모든 행동을 "더 좋게"만 갱신하는 문제가 생깁니다. seed 0에서
비교해봅니다(10.2절의 "솔직한 비고" 표와 동일한 seed).


In [9]:
returns_nonorm = train_reinforce(seed=0, normalize=False)
f2, l2, b2 = stats(returns_nonorm)
print(f"정규화 없음  (seed 0): 처음 20 에피소드 평균={f2:.2f}  "
      f"마지막 20 에피소드 평균={l2:.2f}  최고={b2}")
print()
print("두 설정을 같은 seed(0)로 비교 — seed 0에서는 '없음'이 우연히 잘하는 케이스:")
print("  이게 '정규화 무의미'가 아니라 '300 에피소드로는 분산 감소 이득이")
print("  노이즈에 묻힐 수 있다'는 것(책의 솔직한 비고). 상태별 가치함수")
print("  V(s_t)를 쓰는 베이스라인이 10.3절 Actor-Critic이다.")


정규화 없음  (seed 0): 처음 20 에피소드 평균=19.60  마지막 20 에피소드 평균=383.10  최고=500.0

두 설정을 같은 seed(0)로 비교 — seed 0에서는 '없음'이 우연히 잘하는 케이스:
  이게 '정규화 무의미'가 아니라 '300 에피소드로는 분산 감소 이득이
  노이즈에 묻힐 수 있다'는 것(책의 솔직한 비고). 상태별 가치함수
  V(s_t)를 쓰는 베이스라인이 10.3절 Actor-Critic이다.


### 3.1 두 학습 곡선을 한 그림으로

정규화(베이스라인) 유무의 학습 곡선을 나란히 그려 비교합니다. 개별
에피소드의 노이즈가 얼마나 큰지(= REINFORCE의 고분산)도 한눈에 보입니다.


In [10]:
fig, ax = plt.subplots(figsize=(7, 4))
ep = list(range(1, len(returns_norm)+1))
ax.plot(ep, moving_average(returns_norm, 20), color="tab:blue",
        linewidth=1.6, label="with baseline (mean removed)")
ax.plot(ep, moving_average(returns_nonorm, 20), color="tab:orange",
        linewidth=1.6, label="without baseline (raw G_t)")
ax.axhline(500, color="tab:red", linestyle="--", linewidth=1, label="max score 500")
ax.set_xlabel("Episode"); ax.set_ylabel("Return (20-episode moving average)")
ax.set_title("CartPole: with/without baseline (return-mean subtraction), seed 0")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch10_2_reinforce_baseline_compare.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch10_2_reinforce_baseline_compare.svg")


저장: /home/smhan/book-ml/kor/src/images/ch10_2_reinforce_baseline_compare.svg
